In [3]:
import pandas as pd 
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt
import pennylane as mod

In [5]:
# Constants/parameters

# Number of Qubits we will use, depends on how many photos we will use and stuff
n_qubits = 8

# Number of layers we will use to train the model
n_layers = 3

# Patches: Images will be broken into 4x4 patches; Each image will have 16patches for all 256 pixels
#keep in mind if you want to do larger/smaller images you will have to update the patch sizes
n_patch_size = 4 
n_patches = 16

#random seed set to 42, pls dont change i'll cry
np.random.seed(42)

device = mod.device("default.qubit", wires = n_qubits)

In [7]:
# Variational Circuit
#hi mi llama Abbie

@mod.qnode(device)
def variational_Circuit(inputs, weights):

    #weights for the model, 
    weights = weights.reshape(n_layers, n_qubits, 3)

    #encode patches into quantum state
    for i in range(n_qubits):
        mod.RY(inputs[i], wires=i)
    
    #variational layers
    for layer in range(n_layers):
        #rotation layer
        for qubit in range(n_qubits):
            mod.RX(weights[layer, qubit, 0], wires=qubit)
            mod.RY(weights[layer, qubit, 1], wires=qubit)
            mod.RZ(weights[layer, qubit, 2], wires=qubit)
        
        # Entanglement layer - connect all qubits in a ring
        for qubit in range(n_qubits):
            mod.CNOT(wires=[qubit, (qubit + 1) % n_qubits])
    
    # Return expectation values from each qubit
    return [mod.expval(mod.PauliZ(wires=i)) for i in range(n_qubits)]


In [8]:
#TRAINING AND LOSS FUNCTIONS
def add_gaussian_noise(patch, noise_level):
    #.normal follows a gaussian distribution to generate random samples
    #in our case we use it for noise
    noise = np.random.normal(0, noise_level, size=image.shape)

    #combine the noise to the patch and return
    noisy_patch = patch + noise
    return noisy_patch
